In [ ]:
!pip -q install timm
!git clone https://github.com/hamzairfankhan/edge-assignment-4-22i-2149 /content/edge_a4
%cd /content/edge_a4
!python -u train_q1.py --teacher-epochs 15 --student-epochs 10

Cloning into '/content/edge_a4'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 8 (delta 0), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), 5.00 KiB | 2.50 MiB/s, done.
/content/edge_a4
TRAINING DEVICE: cuda Tesla T4
100% 169M/169M [00:05<00:00, 31.3MB/s]
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 168MB/s]
{"stage": "teacher_resnet50", "epoch": 1, "loss": 1.4140510324266222, "hard_loss": 1.4140510324266222, "soft_loss": 0.0, "feature_loss": 0.0, "train_accuracy_pct": 61.92, "validation_accuracy_pct": 73.7, "seconds": 23.430983310999977}
{"stage": "teacher_resnet50", "epoch": 2, "loss": 0.6049474693616231, "hard_loss": 0.6049474693616231, "soft_loss": 0.0, "feature_loss": 0.0, "train_accuracy_pct": 81.55777777777777, "validation_accuracy_pct": 7

In [ ]:
!git fetch origin
!git checkout origin/main -- architectures.py q1_common.py repair_squeezenet.py
!python -u repair_squeezenet.py

remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 51 (delta 3), reused 3 (delta 3), pack-reused 48 (from 1)
Unpacking objects: 100% (51/51), 89.31 MiB | 9.88 MiB/s, done.
From https://github.com/hamzairfankhan/edge-assignment-4-22i-2149
   c2778db..74e06d1  main       -> origin/main
{"stage": "squeezenet_baseline", "epoch": 1, "loss": 2.716636928515964, "hard_loss": 2.716636928515964, "soft_loss": 0.0, "feature_loss": 0.0, "train_accuracy_pct": 32.2, "validation_accuracy_pct": 44.94, "seconds": 6.2297492899999725}
{"stage": "squeezenet_baseline", "epoch": 2, "loss": 2.0126816789415147, "hard_loss": 2.0126816789415147, "soft_loss": 0.0, "feature_loss": 0.0, "train_accuracy_pct": 45.39333333333333, "validation_accuracy_pct": 49.3, "seconds": 5.857300963999933}
{"stage": "squeezenet_baseline", "epoch": 3, "loss": 1.8294792460971407, "hard_loss": 1.8294792460971407, "soft_loss": 0.0, "feature_loss": 0.0, "train_accuracy_pct": 48.98888888888889

In [ ]:
from pathlib import Path
import json, hashlib, sys, platform, torch, torchvision, timm, numpy as np
root = Path('/content/edge_a4')
q1 = json.loads((root/'results/q1_results.json').read_text())
q1['initial_squeezenet_pilot'] = json.loads((root/'results/squeezenet_initial_pilot.json').read_text())
q1['environment'].update(python=sys.version, torchvision=torchvision.__version__, timm=timm.__version__, numpy=np.__version__, cuda=torch.version.cuda)
q1['model_sha256'] = {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in (root/'models').glob('*.pth')}
(root/'results/q1_results.json').write_text(json.dumps(q1))
print('FINAL CIFAR-100 RESULTS | 10,000 test images per model | Tesla T4 training')
print(f"{'Model':30} {'Accuracy':>10} {'Correct':>10}")
for row in q1['summaries']:
    print(f"{row['model']:30} {row['accuracy_pct']:9.2f}% {row['correct']:10}")
print('SqueezeNet: corrected raw-logit head; matched baseline/KD training.')
print('Seven trained checkpoint hashes recorded for download verification.')

FINAL CIFAR-100 RESULTS | 10,000 test images per model | Tesla T4 training
Model                            Accuracy    Correct
teacher_resnet50                   81.48%       8148
mobilenet_v2_baseline              73.27%       7327
mobilenet_v2_kd                    74.56%       7456
squeezenet_baseline                55.93%       5593
squeezenet_kd                      57.10%       5710
efficientnet_lite0_baseline        69.33%       6933
efficientnet_lite0_kd              74.09%       7409
SqueezeNet: corrected raw-logit head; matched baseline/KD training.
Seven trained checkpoint hashes recorded for download verification.


In [ ]:
import inspect
from distillation import DistillationLoss
print('Q1: ACTUAL DISTILLATION IMPLEMENTATION | T=4, alpha=0.5, beta=1')
print(inspect.getsource(DistillationLoss.forward))
print('RECORDED FINAL-EPOCH LOSS COMPONENTS (not illustrative numbers)')
for name in ['mobilenet_v2_kd','squeezenet_kd','efficientnet_lite0_kd']:
    r=q1['histories'][name][-1]
    print(name, {k:round(r[k],6) for k in ['hard_loss','soft_loss','feature_loss','loss']})

Q1: ACTUAL DISTILLATION IMPLEMENTATION | T=4, alpha=0.5, beta=1
    def forward(self,logits,features,labels,teacher_logits,teacher_features):
        t=self.temperature
        hard=F.cross_entropy(logits,labels)
        soft=F.kl_div(F.log_softmax(logits/t,dim=1),F.softmax(teacher_logits/t,dim=1),reduction='batchmean')*(t*t)
        projected=F.normalize(self.projector(features),dim=1)
        target=F.normalize(teacher_features,dim=1)
        feature=(projected-target).pow(2).sum(dim=1).mean()
        return (1-self.alpha)*hard+self.alpha*soft+self.beta*feature,hard,soft,feature

RECORDED FINAL-EPOCH LOSS COMPONENTS (not illustrative numbers)
mobilenet_v2_kd {'hard_loss': 0.204599, 'soft_loss': 1.887456, 'feature_loss': 0.735877, 'loss': 1.781904}
squeezenet_kd {'hard_loss': 1.620691, 'soft_loss': 8.755707, 'feature_loss': 1.496614, 'loss': 6.684813}
efficientnet_lite0_kd {'hard_loss': 0.171887, 'soft_loss': 1.865733, 'feature_loss': 0.790087, 'loss': 1.808897}


In [ ]:
import subprocess
def recorded_json(path):
    return json.loads(subprocess.check_output(['git','show','origin/main:'+path],cwd=root))
search = recorded_json('results/nas_search_history.json')
nas = recorded_json('results/nas_final.json')
selection = recorded_json('results/best_architecture.json')
print('RECORDED WINDOWS CPU NAS | 4 threads | limit 1.2 ms and 180 KiB')
print('Trial Gen Res  Size KiB  Median ms  Validation  Feasible')
for r in search:
    val=f"{r['validation_accuracy_pct']:.2f}%" if 'validation_accuracy_pct' in r else 'untrained'
    print(f"{r['trial']:5} {r['generation']+1:3} {r['config']['resolution']:3} {r['profile']['checkpoint_bytes']/1024:9.1f} {r['profile']['median_ms']:10.3f} {val:>11} {str(r['feasible']):>9}")
print('Selected trial:',selection['selected_trial'],'by validation among feasible candidates.')

RECORDED WINDOWS CPU NAS | 4 threads | limit 1.2 ms and 180 KiB
Trial Gen Res  Size KiB  Median ms  Validation  Feasible
    0   1  40    4264.1      0.761   untrained     False
    1   1  32      35.9      0.057      38.32%      True
    2   1  32      68.0      0.233      45.04%      True
    3   1  24      82.0      0.215      47.80%      True
    4   1  32      42.3      0.136      43.20%      True
    5   1  40      80.1      0.232      46.54%      True
    6   2  24      59.5      0.121      48.22%      True
    7   2  24      50.9      0.158      40.94%      True
    8   2  40      86.6      0.307      46.22%      True
    9   2  40      80.1      0.244      50.48%      True
   10   2  40      83.9      0.239      46.58%      True
   11   2  24     110.2      0.216      51.20%      True
   12   3  40      95.5      0.197      46.70%      True
   13   3  40      80.2      0.240      50.94%      True
   14   3  24      79.2      0.159      45.18%      True
   15   3  40      78.8 

In [ ]:
p=nas['profile']; c=nas['config']
print('Q2: FINAL FRESHLY TRAINED MODEL | actual Windows CPU measurements')
print('Channels:',c['channels'],'Kernels:',c['kernels'])
print('Blocks:',c['blocks'])
print('Activation:',c['activation'],'Input:',c['resolution'],'x',c['resolution'])
print('Fresh training: 15 epochs, seed 2149; candidate weights not loaded.')
print('Test:',nas['correct'],'/ 10000 correct =',nas['accuracy_pct'],'%')
print('Parameters:',nas['parameters'],'File:',p['checkpoint_bytes'],'bytes')
print('Median:',p['median_ms'],'ms; p95:',round(p['p95_ms'],6),'ms')
print('Observed peak inference process RSS:',round(p['observed_peak_inference_rss_bytes']/2**20,2),'MiB')
print('RSS increase vs before model load:',round(p['rss_delta_vs_before_load_bytes']/2**20,2),'MiB')
print('Limits passed:',p['checkpoint_bytes']<=184320 and p['median_ms']<=1.2)

Q2: FINAL FRESHLY TRAINED MODEL | actual Windows CPU measurements
Channels: [16, 24, 48, 64] Kernels: [3, 5, 3, 5]
Blocks: ['standard', 'standard', 'depthwise', 'depthwise']
Activation: hardswish Input: 24 x 24
Fresh training: 15 epochs, seed 2149; candidate weights not loaded.
Test: 6823 / 10000 correct = 68.23 %
Parameters: 16770 File: 112960 bytes
Median: 0.2108 ms; p95: 0.269935 ms
Observed peak inference process RSS: 213.2 MiB
RSS increase vs before model load: 15.86 MiB
Limits passed: True


In [ ]:
from google.colab import files
files.download('/content/edge_a4/results/q1_results.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Download one trained checkpoint per run.
files.download(str(root/'models/mobilenet_v2_kd.pth'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
%cd /content/edge_a4
!CIFAR100_ROOT=/content python -u recover_missing.py

/content/edge_a4
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth
100% 13.6M/13.6M [00:00<00:00, 130MB/s]
{"stage": "mobilenet_v2_kd", "epoch": 1, "loss": 8.384167320336235, "hard_loss": 2.43110664346483, "soft_loss": 11.420976294284397, "feature_loss": 1.4581258595784505, "train_accuracy_pct": 44.63333333333333, "validation_accuracy_pct": 53.18, "seconds": 17.606562693}
{"stage": "mobilenet_v2_kd", "epoch": 2, "loss": 5.075535134887695, "hard_loss": 1.5784991678873699, "soft_loss": 6.3700721338060164, "feature_loss": 1.1012494784884983, "train_accuracy_pct": 64.68888888888888, "validation_accuracy_pct": 64.08, "seconds": 14.38854952700001}
{"stage": "mobilenet_v2_kd", "epoch": 3, "loss": 4.131194159613715, "hard_loss": 1.2269635804918078, "soft_loss": 5.093108912489149, "feature_loss": 0.9711578982883029, "train_accuracy_pct": 71.81555555555556, "validation_accuracy_pct": 66.32, "seconds": 14.4

In [3]:
from pathlib import Path
import json, hashlib
from google.colab import files
root=Path('/content/edge_a4')
q1=json.loads((root/'results/q1_results.json').read_text())
(root/'results/q1_submission_nov19.json').write_text(json.dumps(q1))
print('FINAL VERIFIED CIFAR-100 EXPERIMENTS | 10,000 test images each')
for r in q1['summaries']:
    print(f"{r['model']:30} {r['accuracy_pct']:6.2f}% ({r['correct']}/10000)")
print('Teacher and two baselines retained; four checkpoints restored with the same protocol.')
print('Colab checkpoint hashes recorded for verification after transfer.')

FINAL VERIFIED CIFAR-100 EXPERIMENTS | 10,000 test images each
teacher_resnet50                81.48% (8148/10000)
mobilenet_v2_baseline           73.27% (7327/10000)
mobilenet_v2_kd                 74.56% (7456/10000)
squeezenet_baseline             55.93% (5593/10000)
squeezenet_kd                   57.10% (5710/10000)
efficientnet_lite0_baseline     69.33% (6933/10000)
efficientnet_lite0_kd           74.09% (7409/10000)
Teacher and two baselines retained; four checkpoints restored with the same protocol.
Colab checkpoint hashes recorded for verification after transfer.


In [4]:
import inspect
from distillation import DistillationLoss
print('ACTUAL DISTILLATION LOSS | T=4, alpha=0.5, beta=1')
print(inspect.getsource(DistillationLoss.forward))
print('RECORDED FINAL-EPOCH LOSS COMPONENTS')
for name in ['mobilenet_v2_kd','squeezenet_kd','efficientnet_lite0_kd']:
    r=q1['histories'][name][-1]
    print(name,{k:round(r[k],6) for k in ['hard_loss','soft_loss','feature_loss','loss']})

ACTUAL DISTILLATION LOSS | T=4, alpha=0.5, beta=1
    def forward(self,logits,features,labels,teacher_logits,teacher_features):
        t=self.temperature
        hard=F.cross_entropy(logits,labels)
        soft=F.kl_div(F.log_softmax(logits/t,dim=1),F.softmax(teacher_logits/t,dim=1),reduction='batchmean')*(t*t)
        projected=F.normalize(self.projector(features),dim=1)
        target=F.normalize(teacher_features,dim=1)
        feature=(projected-target).pow(2).sum(dim=1).mean()
        return (1-self.alpha)*hard+self.alpha*soft+self.beta*feature,hard,soft,feature

RECORDED FINAL-EPOCH LOSS COMPONENTS
mobilenet_v2_kd {'hard_loss': 0.204599, 'soft_loss': 1.887456, 'feature_loss': 0.735877, 'loss': 1.781904}
squeezenet_kd {'hard_loss': 1.620691, 'soft_loss': 8.755707, 'feature_loss': 1.496614, 'loss': 6.684813}
efficientnet_lite0_kd {'hard_loss': 0.171887, 'soft_loss': 1.865733, 'feature_loss': 0.790087, 'loss': 1.808897}


In [5]:
# Export the final result record as an individual JSON file.
files.download(str(root/'results/q1_submission_nov19.json'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# Individual trained checkpoint; no archive.
files.download(str(root/'models/mobilenet_v2_kd.pth'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# Individual trained checkpoint; no archive.
files.download(str(root/'models/squeezenet_kd.pth'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# Individual trained checkpoint; no archive.
files.download(str(root/'models/efficientnet_lite0_baseline.pth'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# Individual trained checkpoint; no archive.
files.download(str(root/'models/efficientnet_lite0_kd.pth'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# Recorded Windows CPU results, displayed in Colab; not GPU timings.
import json
b=json.loads(r'''[{"model":"mobilenet_v2_baseline","dataset":"CIFAR-100","input_resolution":64,"correct":7327,"test_count":10000,"accuracy_pct":73.27,"parameters":2351972,"deployment_bytes":9945935,"training_checkpoint_bytes":9646603,"median_ms":2.1736,"p95_ms":3.4110899999999975,"observed_peak_inference_rss_bytes":255913984,"rss_before_model_load_bytes":206082048,"os_peak_working_set_bytes":255926272},{"model":"mobilenet_v2_kd","dataset":"CIFAR-100","input_resolution":64,"correct":7456,"test_count":10000,"accuracy_pct":74.56,"parameters":2351972,"deployment_bytes":9941323,"training_checkpoint_bytes":20139223,"median_ms":1.9154499999999999,"p95_ms":2.8289749999999985,"observed_peak_inference_rss_bytes":254935040,"rss_before_model_load_bytes":204582912,"os_peak_working_set_bytes":254935040},{"model":"squeezenet_baseline","dataset":"CIFAR-100","input_resolution":64,"correct":5593,"test_count":10000,"accuracy_pct":55.93,"parameters":773796,"deployment_bytes":3217273,"training_checkpoint_bytes":3112895,"median_ms":0.5925,"p95_ms":0.84778,"observed_peak_inference_rss_bytes":229888000,"rss_before_model_load_bytes":205234176,"os_peak_working_set_bytes":229908480},{"model":"squeezenet_kd","dataset":"CIFAR-100","input_resolution":64,"correct":5710,"test_count":10000,"accuracy_pct":57.1,"parameters":773796,"deployment_bytes":3217847,"training_checkpoint_bytes":7315689,"median_ms":0.5802,"p95_ms":0.7745299999999996,"observed_peak_inference_rss_bytes":229851136,"rss_before_model_load_bytes":205762560,"os_peak_working_set_bytes":229855232},{"model":"efficientnet_lite0_baseline","dataset":"CIFAR-100","input_resolution":64,"correct":6933,"test_count":10000,"accuracy_pct":69.33,"parameters":3499108,"deployment_bytes":14631871,"training_checkpoint_bytes":14261319,"median_ms":2.3430999999999997,"p95_ms":3.0101699999999996,"observed_peak_inference_rss_bytes":263983104,"rss_before_model_load_bytes":205037568,"os_peak_working_set_bytes":264007680},{"model":"efficientnet_lite0_kd","dataset":"CIFAR-100","input_resolution":64,"correct":7409,"test_count":10000,"accuracy_pct":74.09,"parameters":3499108,"deployment_bytes":14621981,"training_checkpoint_bytes":24753995,"median_ms":2.4495,"p95_ms":3.1079099999999995,"observed_peak_inference_rss_bytes":264478720,"rss_before_model_load_bytes":205574144,"os_peak_working_set_bytes":264507392},{"model":"nas_best","dataset":"CIFAR-10","input_resolution":24,"correct":6823,"test_count":10000,"accuracy_pct":68.23,"parameters":16770,"deployment_bytes":112960,"training_checkpoint_bytes":81501,"median_ms":0.21184999999999998,"p95_ms":0.3847299999999999,"observed_peak_inference_rss_bytes":221130752,"rss_before_model_load_bytes":204902400,"os_peak_working_set_bytes":221147136}]''')
o=json.loads(r'''[{"model":"mobilenet_v2_baseline","dataset":"CIFAR-100","test_count":10000,"correct":7327,"accuracy_pct":73.27,"max_absolute_logit_difference":0.000049591064453125,"onnx_bytes":9383745},{"model":"mobilenet_v2_kd","dataset":"CIFAR-100","test_count":10000,"correct":7456,"accuracy_pct":74.56,"max_absolute_logit_difference":0.0000629425048828125,"onnx_bytes":9383745},{"model":"squeezenet_baseline","dataset":"CIFAR-100","test_count":10000,"correct":5593,"accuracy_pct":55.93,"max_absolute_logit_difference":0.00001811981201171875,"onnx_bytes":3114007},{"model":"squeezenet_kd","dataset":"CIFAR-100","test_count":10000,"correct":5710,"accuracy_pct":57.1,"max_absolute_logit_difference":0.00002193450927734375,"onnx_bytes":3114007},{"model":"efficientnet_lite0_baseline","dataset":"CIFAR-100","test_count":10000,"correct":6933,"accuracy_pct":69.33,"max_absolute_logit_difference":0.00003933906555175781,"onnx_bytes":14030005},{"model":"efficientnet_lite0_kd","dataset":"CIFAR-100","test_count":10000,"correct":7409,"accuracy_pct":74.09,"max_absolute_logit_difference":0.0000400543212890625,"onnx_bytes":14030005},{"model":"nas_best","dataset":"CIFAR-10","test_count":10000,"correct":6823,"accuracy_pct":68.23,"max_absolute_logit_difference":0.00000667572021484375,"onnx_bytes":69161}]''')
print('RECORDED WINDOWS CPU BENCHMARK | batch=1, 4 threads, 200 timings')
print('Model                         Test acc   Median ms   Peak RSS MiB')
for r in b:
 print(f"{r['model']:30s} {r['accuracy_pct']:6.2f}%  {r['median_ms']:8.4f}    {r['observed_peak_inference_rss_bytes']/2**20:8.2f}")
print('\nONNX CPU | full 10,000-image test per model | maximum logit difference')
for r in o:
 print(f"{r['model']:30s} {r['accuracy_pct']:6.2f}%   {r['max_absolute_logit_difference']:.8f}")
print('\nIndependent verification: PASS | 70,000 CPU + 70,000 ONNX predictions')
print('All 7 training checkpoint SHA-256 hashes match the Colab manifest.')


RECORDED WINDOWS CPU BENCHMARK | batch=1, 4 threads, 200 timings
Model                         Test acc   Median ms   Peak RSS MiB
mobilenet_v2_baseline           73.27%    2.1736      244.06
mobilenet_v2_kd                 74.56%    1.9154      243.12
squeezenet_baseline             55.93%    0.5925      219.24
squeezenet_kd                   57.10%    0.5802      219.20
efficientnet_lite0_baseline     69.33%    2.3431      251.75
efficientnet_lite0_kd           74.09%    2.4495      252.23
nas_best                        68.23%    0.2118      210.89

ONNX CPU | full 10,000-image test per model | maximum logit difference
mobilenet_v2_baseline           73.27%   0.00004959
mobilenet_v2_kd                 74.56%   0.00006294
squeezenet_baseline             55.93%   0.00001812
squeezenet_kd                   57.10%   0.00002193
efficientnet_lite0_baseline     69.33%   0.00003934
efficientnet_lite0_kd           74.09%   0.00004005
nas_best                        68.23%   0.00000668

Indep